# 🎬 LTX-2 Distilled · SVI-Pro · Director 2.0 · Google Colab Pipeline

**Production-quality Image-to-Video pipeline for Google Colab T4 GPU**

| Feature | Spec |
|---|---|
| **Model** | LTX-2 Distilled GGUF Q4_K_M (19B) |
| **Max Resolution** | 480p |
| **Max Duration** | 4 seconds |
| **GPU Target** | Tesla T4 (16 GB VRAM) |
| **Modes** | Text-to-Video · Image-to-Video · Multi-Scene Story |

### 📋 Pipeline Sections
1. Environment Setup   2. Dependency Installation   3. ComfyUI Installation  
4. Custom Nodes   5. Model Downloader   6. Workflow Parser  
7. Runtime Config   8. Story Config   9. Character Database  
10. Prompt Compiler   11. Director Guidance   12. Latent Generation  
13. Sampler   14. Decoder   15. Video Export   16. Preview   17. Cleanup

---
> **References:** `ltx2_ti2v_distilled-.py` · `LTX-2.3_Director_2.0-MV-Workflow-30s.json` · `SVI-Pro-Workflow.json`

# ⚡ Section 1 · Environment Setup

In [ ]:
# @title ⚡ 1. Environment Setup { display-mode: "form" }
# @markdown Sets CUDA allocator, verifies GPU, installs system packages.

import os, subprocess, sys, time, gc
from pathlib import Path

# ── CUDA allocator: reduces fragmentation on 16 GB T4 ──────────────────────
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# ── GPU probe ──────────────────────────────────────────────────────────────
def probe_gpu() -> dict:
    """Return VRAM stats and GPU name. Prints a summary."""
    try:
        import torch
        if not torch.cuda.is_available():
            print('⚠️  No CUDA GPU detected. Switch Runtime → T4 GPU.')
            return {}
        props = torch.cuda.get_device_properties(0)
        total_gb = props.total_memory / 1e9
        free_gb  = (props.total_memory - torch.cuda.memory_allocated(0)) / 1e9
        info = {'name': props.name, 'total_gb': total_gb, 'free_gb': free_gb}
        print(f'✅ GPU  : {props.name}')
        print(f'   VRAM : {total_gb:.1f} GB total  |  {free_gb:.1f} GB free')
        if total_gb < 14:
            print('⚠️  Less than 14 GB VRAM — pipeline may OOM at full settings.')
        return info
    except Exception as e:
        print(f'GPU probe failed: {e}')
        return {}

# ── System packages ────────────────────────────────────────────────────────
def install_system_packages() -> None:
    """Install aria2 (fast downloader) and ffmpeg (video mux)."""
    pkgs = ['aria2', 'ffmpeg']
    try:
        result = subprocess.run(
            ['apt-get', '-y', 'install', '-qq'] + pkgs,
            check=True, capture_output=True
        )
        print(f'✅ System packages: {pkgs}')
    except subprocess.CalledProcessError as e:
        print(f'⚠️  apt error: {e.stderr.decode().strip()}')

print('📦 Installing system packages...')
install_system_packages()

print('\n🔍 GPU Status:')
GPU_INFO = probe_gpu()

print('\n✅ Section 1 complete.')

# 📦 Section 2 · Dependency Installation

In [ ]:
# @title 📦 2. Dependency Installation { display-mode: "form" }
# @markdown Installs PyTorch, diffusers, einops, spandrel, opencv, imageio.

from IPython.display import clear_output

def pip_install(packages: list, quiet: bool = True) -> None:
    """Install pip packages with optional quiet flag."""
    flags = ['-q'] if quiet else []
    cmd = [sys.executable, '-m', 'pip', 'install'] + flags + packages
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f'⚠️  pip error: {result.stderr[:500]}')

print('🔧 Installing PyTorch stack...')
pip_install(['torch', 'torchvision', 'torchaudio'])

print('🔧 Installing diffusion & video libraries...')
pip_install([
    'torchsde', 'einops', 'diffusers', 'accelerate',
    'av', 'spandrel', 'albumentations',
    'onnx', 'opencv-python', 'onnxruntime',
    'imageio', 'imageio-ffmpeg',
    'nest_asyncio', 'tqdm', 'psutil',
])

clear_output()

# ── Verify core imports ────────────────────────────────────────────────────
import torch, numpy as np, cv2
from PIL import Image
import imageio, gc, shutil, json, re
from typing import Sequence, Mapping, Any, Union, Optional, List, Dict
import psutil

print(f'✅ PyTorch  : {torch.__version__}')
print(f'   CUDA     : {torch.version.cuda}')
print(f'   Device   : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print('\n✅ Section 2 complete.')

# 🖥️ Section 3 · ComfyUI Installation

In [ ]:
# @title 🖥️ 3. ComfyUI Installation { display-mode: "form" }
# @markdown Clones the pinned ComfyUI release used by the reference workflow.

COMFYUI_DIR = Path('/content/ComfyUI')
COMFYUI_REPO = 'https://github.com/Isi-dev/ComfyUI.git'
COMFYUI_BRANCH = 'ComfyUI_22_01_2026_v0.10.0'

def clone_repo(url: str, dest: Path, branch: str = None, depth: int = 1) -> bool:
    """Clone a git repo. Returns True on success."""
    if dest.exists():
        print(f'  ↩️  Already exists: {dest.name} — skipping clone.')
        return True
    cmd = ['git', 'clone', '--depth', str(depth)]
    if branch:
        cmd += ['--branch', branch]
    cmd += [url, str(dest)]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f'  ❌ Clone failed: {result.stderr[:300]}')
        return False
    print(f'  ✅ Cloned: {dest.name}')
    return True

print('⬇️  Cloning ComfyUI...')
clone_repo(COMFYUI_REPO, COMFYUI_DIR, branch=COMFYUI_BRANCH)

print('\n📦 Installing ComfyUI requirements...')
pip_install(['-r', str(COMFYUI_DIR / 'requirements.txt')])

# ── Add ComfyUI to Python path ─────────────────────────────────────────────
if str(COMFYUI_DIR) not in sys.path:
    sys.path.insert(0, str(COMFYUI_DIR))

os.makedirs(COMFYUI_DIR / 'models' / 'unet',              exist_ok=True)
os.makedirs(COMFYUI_DIR / 'models' / 'text_encoders',     exist_ok=True)
os.makedirs(COMFYUI_DIR / 'models' / 'vae',               exist_ok=True)
os.makedirs(COMFYUI_DIR / 'models' / 'loras',             exist_ok=True)
os.makedirs(COMFYUI_DIR / 'models' / 'latent_upscale_models', exist_ok=True)
os.makedirs(COMFYUI_DIR / 'input',                        exist_ok=True)
os.makedirs(COMFYUI_DIR / 'output',                       exist_ok=True)

clear_output()
print('✅ ComfyUI installed.')
print(f'   Path: {COMFYUI_DIR}')
print('\n✅ Section 3 complete.')

# 🔌 Section 4 · Custom Node Installation

In [ ]:
# @title 🔌 4. Custom Node Installation { display-mode: "form" }
# @markdown Installs KJNodes (audio VAE + utilities) and ComfyUI-GGUF (GGUF loader).

CUSTOM_NODES_DIR = COMFYUI_DIR / 'custom_nodes'
os.makedirs(CUSTOM_NODES_DIR, exist_ok=True)

CUSTOM_NODES = [
    {
        'name': 'ComfyUI_KJNodes',
        'url': 'https://github.com/Isi-dev/ComfyUI_KJNodes',
        'branch': 'kj_1.2.6',
        'requirements': True,
    },
    {
        'name': 'ComfyUI_GGUF',
        'url': 'https://github.com/Isi-dev/ComfyUI_GGUF.git',
        'branch': 'ComfyUI_GGUF_22_01_2026',
        'requirements': True,
    },
]

for node in CUSTOM_NODES:
    dest = CUSTOM_NODES_DIR / node['name']
    print(f'\n⬇️  {node["name"]}...')
    cloned = clone_repo(node['url'], dest, branch=node.get('branch'))
    if cloned and node.get('requirements'):
        req = dest / 'requirements.txt'
        if req.exists():
            print(f'  📦 Installing requirements for {node["name"]}...')
            pip_install(['-r', str(req)])

clear_output()
print('✅ Custom nodes installed:')
for node in CUSTOM_NODES:
    status = '✓' if (CUSTOM_NODES_DIR / node['name']).exists() else '✗'
    print(f'   {status}  {node["name"]}')
print('\n✅ Section 4 complete.')

# ⬇️ Section 5 · Model Downloader

In [ ]:
# @title ⬇️ 5. Model Downloader { display-mode: "form" }
# @markdown Downloads all required models: GGUF DiT, text encoders, VAEs, upscaler, LoRAs.

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  MODEL URLS — edit these to swap variants                              ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# DiT (Diffusion Transformer) — GGUF Q4_K_M
LTX_MODEL_URL       = 'https://huggingface.co/Kijai/LTXV2_comfy/resolve/main/diffusion_models/ltx-2-19b-distilled_Q4_K_M.gguf'  # @param {type:"string"}
# Text encoder 1 — Gemma 3 12B fp4
TEXT_ENC1_URL       = 'https://huggingface.co/Comfy-Org/ltx-2/resolve/main/split_files/text_encoders/gemma_3_12B_it_fp4_mixed.safetensors'  # @param {type:"string"}
# Text encoder 2 — LTX-2 embeddings connector
TEXT_ENC2_URL       = 'https://huggingface.co/Kijai/LTXV2_comfy/resolve/main/text_encoders/ltx-2-19b-embeddings_connector_distill_bf16.safetensors'  # @param {type:"string"}
# Video VAE
VAE_VIDEO_URL       = 'https://huggingface.co/Kijai/LTXV2_comfy/resolve/main/VAE/LTX2_video_vae_bf16.safetensors'  # @param {type:"string"}
# Audio VAE
VAE_AUDIO_URL       = 'https://huggingface.co/Kijai/LTXV2_comfy/resolve/main/VAE/LTX2_audio_vae_bf16.safetensors'  # @param {type:"string"}
# Spatial upscaler x2
UPSCALER_URL        = 'https://huggingface.co/Lightricks/LTX-2/resolve/main/ltx-2-spatial-upscaler-x2-1.0.safetensors'  # @param {type:"string"}

# ── LoRA Downloads ─────────────────────────────────────────────────────────
# LoRA 1: Camera control (Dolly-Left example — swap to your preferred LoRA)
DOWNLOAD_LORA_1 = False  # @param {type:"boolean"}
LORA_1_URL      = 'https://huggingface.co/Lightricks/LTX-2-19b-LoRA-Camera-Control-Dolly-Left/resolve/main/ltx-2-19b-lora-camera-control-dolly-left.safetensors'  # @param {type:"string"}
LORA_1_STRENGTH = 1.0  # @param {type:"number"}

DOWNLOAD_LORA_2 = False  # @param {type:"boolean"}
LORA_2_URL      = ''  # @param {type:"string"}
LORA_2_STRENGTH = 1.0  # @param {type:"number"}

DOWNLOAD_LORA_3 = False  # @param {type:"boolean"}
LORA_3_URL      = ''  # @param {type:"string"}
LORA_3_STRENGTH = 1.0  # @param {type:"number"}

CIVITAI_TOKEN   = ''  # @param {type:"string"}

# ══════════════════════════════════════════════════════════════════════════

def model_download_aria2(
    url: str,
    dest_dir: str,
    filename: str = None,
    silent: bool = True,
    max_retries: int = 3,
) -> Optional[str]:
    """
    Download a file with aria2c.

    Uses 16 parallel connections and 16 MB chunks for maximum Colab throughput.
    Supports resume (aria2 -c flag). Retries up to max_retries times.

    Args:
        url        : Direct download URL.
        dest_dir   : Destination directory (created if absent).
        filename   : Override filename; defaults to last URL segment.
        silent     : Suppress aria2 console output.
        max_retries: Number of retry attempts.

    Returns:
        Filename string on success, None on failure.
    """
    Path(dest_dir).mkdir(parents=True, exist_ok=True)
    if filename is None:
        filename = url.split('/')[-1].split('?')[0]

    dest_path = Path(dest_dir) / filename
    if dest_path.exists() and dest_path.stat().st_size > 1_000_000:
        print(f'  ↩️  Cached: {filename}')
        return filename

    cmd = [
        'aria2c',
        '--console-log-level=error',
        '-c', '-x', '16', '-s', '16', '-k', '1M',
        '--max-tries', str(max_retries),
        '--retry-wait', '5',
        '-d', str(dest_dir),
        '-o', filename,
        url,
    ]
    if silent:
        cmd.extend(['--summary-interval=0', '--quiet'])

    print(f'  ⬇️  {filename} ...', end=' ', flush=True)
    t0 = time.time()
    for attempt in range(1, max_retries + 1):
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode == 0:
            elapsed = time.time() - t0
            size_mb = dest_path.stat().st_size / 1e6 if dest_path.exists() else 0
            print(f'Done  ({size_mb:.0f} MB, {elapsed:.0f}s)')
            return filename
        print(f'retry {attempt}...', end=' ', flush=True)
        time.sleep(5)

    print(f'FAILED')
    print(f'     Error: {result.stderr.strip()[:200]}')
    return None


def download_civitai_model(
    url: str, token: str, dest_dir: str = str(COMFYUI_DIR / 'models/loras')
) -> Optional[str]:
    """Download a model from CivitAI using wget with auth token."""
    Path(dest_dir).mkdir(parents=True, exist_ok=True)
    try:
        model_id = url.split('/models/')[1].split('?')[0]
    except IndexError:
        raise ValueError(f'Invalid CivitAI URL: {url}')

    dl_url = f'https://civitai.com/api/download/models/{model_id}?type=Model&format=SafeTensor'
    if token:
        dl_url += f'&token={token}'

    filename = f'civitai_{model_id}.safetensors'
    dest_path = Path(dest_dir) / filename

    cmd = f'wget --max-redirect=10 --show-progress "{dl_url}" -O "{dest_path}"'
    print(f'  ⬇️  CivitAI model {model_id}...')
    os.system(cmd)

    if dest_path.exists() and dest_path.stat().st_size > 0:
        print(f'  ✅ CivitAI download complete: {filename}')
        return filename
    print(f'  ❌ CivitAI download failed.')
    return None


def download_lora(
    url: str,
    dest_dir: str = str(COMFYUI_DIR / 'models/loras'),
    civitai_token: str = None,
) -> Optional[str]:
    """Route download to aria2c (HuggingFace) or CivitAI handler."""
    if not url:
        return None
    if 'civitai.com' in url.lower():
        if not civitai_token:
            raise ValueError('CivitAI token required for CivitAI downloads.')
        return download_civitai_model(url, civitai_token, dest_dir)
    return model_download_aria2(url, dest_dir)


def verify_model(dest_dir: str, filename: str, min_mb: float = 10.0) -> bool:
    """Check a downloaded file exists and exceeds minimum size."""
    if not filename:
        return False
    p = Path(dest_dir) / filename
    if not p.exists():
        return False
    return p.stat().st_size > min_mb * 1e6


# ── Execute downloads ──────────────────────────────────────────────────────
MODEL_DIR    = str(COMFYUI_DIR / 'models/unet')
TEXTENC_DIR  = str(COMFYUI_DIR / 'models/text_encoders')
VAE_DIR      = str(COMFYUI_DIR / 'models/vae')
LORA_DIR     = str(COMFYUI_DIR / 'models/loras')
UPSCALE_DIR  = str(COMFYUI_DIR / 'models/latent_upscale_models')

print('=' * 60)
print('  MODEL DOWNLOADS')
print('=' * 60)

print('\n[1/6] DiT transformer (GGUF):')
DIT_MODEL = model_download_aria2(LTX_MODEL_URL, MODEL_DIR)

print('\n[2/6] Text Encoder 1 (Gemma fp4):')
TEXT_ENC1_MODEL = model_download_aria2(TEXT_ENC1_URL, TEXTENC_DIR)

print('\n[3/6] Text Encoder 2 (Embeddings connector):')
TEXT_ENC2_MODEL = model_download_aria2(TEXT_ENC2_URL, TEXTENC_DIR)

print('\n[4/6] Video VAE:')
VAE_VIDEO_MODEL = model_download_aria2(VAE_VIDEO_URL, VAE_DIR)

print('\n[5/6] Audio VAE:')
VAE_AUDIO_MODEL = model_download_aria2(VAE_AUDIO_URL, VAE_DIR)

print('\n[6/6] Spatial Upscaler x2:')
UPSCALER_MODEL = model_download_aria2(UPSCALER_URL, UPSCALE_DIR)

# LoRAs
VALID_LORA_EXTS = {'.safetensors', '.ckpt', '.pt', '.pth', '.sft'}

def _check_lora(filename):
    if filename and not any(filename.lower().endswith(e) for e in VALID_LORA_EXTS):
        print(f'  ❌ Invalid LoRA format: {filename}')
        return None
    return filename

LORA_1 = LORA_2 = LORA_3 = None
if DOWNLOAD_LORA_1:
    print('\n[LoRA 1]:')
    LORA_1 = _check_lora(download_lora(LORA_1_URL, LORA_DIR, CIVITAI_TOKEN))
if DOWNLOAD_LORA_2:
    print('\n[LoRA 2]:')
    LORA_2 = _check_lora(download_lora(LORA_2_URL, LORA_DIR, CIVITAI_TOKEN))
if DOWNLOAD_LORA_3:
    print('\n[LoRA 3]:')
    LORA_3 = _check_lora(download_lora(LORA_3_URL, LORA_DIR, CIVITAI_TOKEN))

# ── Manifest summary ───────────────────────────────────────────────────────
print('\n' + '=' * 60)
print('  DOWNLOAD MANIFEST')
print('=' * 60)
manifest = [
    ('DiT (GGUF)',        MODEL_DIR,   DIT_MODEL),
    ('Text Encoder 1',   TEXTENC_DIR, TEXT_ENC1_MODEL),
    ('Text Encoder 2',   TEXTENC_DIR, TEXT_ENC2_MODEL),
    ('Video VAE',        VAE_DIR,     VAE_VIDEO_MODEL),
    ('Audio VAE',        VAE_DIR,     VAE_AUDIO_MODEL),
    ('Upscaler',         UPSCALE_DIR, UPSCALER_MODEL),
]
for label, d, f in manifest:
    ok = '✅' if verify_model(d, f) else '❌'
    print(f'  {ok} {label:20s}: {f}')

print('\n✅ Section 5 complete.')

# 🔍 Section 6 · Workflow Parser & ComfyUI Node Loader

In [ ]:
# @title 🔍 6. Workflow Parser & ComfyUI Node Loader { display-mode: "form" }
# @markdown Loads all ComfyUI nodes used in the workflow. Each ComfyUI node is
# @markdown wrapped in a typed Python class for clean pipeline composition.

import asyncio
import nest_asyncio

# ── Helpers (preserved from reference implementation) ─────────────────────

def get_value_at_index(obj: Union[Sequence, Mapping], index: int) -> Any:
    """
    Retrieve the value at position `index` from a ComfyUI node output.

    ComfyUI nodes return either a plain tuple/list or a dict with a 'result'
    key. This helper normalises both cases.
    """
    try:
        return obj[index]
    except KeyError:
        return obj['result'][index]


def tensor_width_height(image) -> tuple:
    """
    Return (width, height) for a ComfyUI image tensor (NHWC or HWC).

    Works without GetImageSize node so it is safe on all ComfyUI builds.
    """
    if isinstance(image, (tuple, list)):
        image = get_value_at_index(image, 0)
    if image.ndim == 4:   # (N, H, W, C)
        return int(image.shape[2]), int(image.shape[1])
    if image.ndim == 3:   # (H, W, C)
        return int(image.shape[1]), int(image.shape[0])
    raise ValueError(f'Unsupported image tensor shape: {image.shape}')


# ── Node loader ────────────────────────────────────────────────────────────

def import_custom_nodes() -> None:
    """
    Load all built-in and external ComfyUI custom nodes.

    Uses nest_asyncio to allow running inside the Jupyter event loop.
    """
    from nodes import init_builtin_extra_nodes, init_external_custom_nodes

    async def _loader():
        failed = await init_builtin_extra_nodes()
        await init_external_custom_nodes()
        if failed:
            print('⚠️  Some extra nodes failed to import:')
            for n in failed:
                print(f'   - {n}')

    try:
        asyncio.run(_loader())
    except RuntimeError:
        nest_asyncio.apply()
        loop = asyncio.get_event_loop()
        loop.run_until_complete(_loader())


def load_audio_vae_compat(vae_name: str):
    """
    Load the LTX audio VAE with KJNodes VAELoaderKJ (preferred) or
    fall back to the built-in VAELoader for older ComfyUI builds.
    """
    from nodes import NODE_CLASS_MAPPINGS
    if 'VAELoaderKJ' in NODE_CLASS_MAPPINGS:
        loader = NODE_CLASS_MAPPINGS['VAELoaderKJ']()
        return loader.load_vae(vae_name=vae_name, device='main_device', weight_dtype='fp16')
    if 'VAELoader' in NODE_CLASS_MAPPINGS:
        loader = NODE_CLASS_MAPPINGS['VAELoader']()
        return loader.load_vae(vae_name=vae_name)
    raise KeyError('No compatible VAE loader found.')


# ── Workflow JSON parser ───────────────────────────────────────────────────

def parse_workflow_json(json_path: str) -> dict:
    """
    Parse a ComfyUI workflow JSON file and return a node-indexed dict.

    Each entry contains: type, widgets_values, inputs (links), and order.
    This is used for introspection / logging; actual execution goes through
    the typed Python pipeline below.
    """
    with open(json_path, 'r') as f:
        data = json.load(f)

    nodes = {}
    for node in data.get('nodes', []):
        node_id = node.get('id')
        nodes[node_id] = {
            'type':           node.get('type', ''),
            'title':          node.get('title', node.get('type', '')),
            'widgets_values': node.get('widgets_values', []),
            'inputs':         [inp.get('name') for inp in node.get('inputs', [])],
            'outputs':        [out.get('name') for out in node.get('outputs', [])],
            'order':          node.get('order', 0),
        }
    return nodes


# ── Memory helpers ─────────────────────────────────────────────────────────

def clear_gpu_memory() -> None:
    """Aggressive GPU + CPU memory reclaim after each pipeline stage."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def log_memory(label: str = '') -> dict:
    """Log current RAM and VRAM usage. Returns a stats dict."""
    stats = {}
    ram = psutil.virtual_memory()
    stats['ram_used_gb']  = ram.used / 1e9
    stats['ram_total_gb'] = ram.total / 1e9
    if torch.cuda.is_available():
        alloc = torch.cuda.memory_allocated(0) / 1e9
        reserved = torch.cuda.memory_reserved(0) / 1e9
        stats['vram_alloc_gb']    = alloc
        stats['vram_reserved_gb'] = reserved
        tag = f'[{label}] ' if label else ''
        print(f'  {tag}RAM {stats["ram_used_gb"]:.1f}/{stats["ram_total_gb"]:.0f} GB  '
              f'| VRAM alloc {alloc:.2f} GB  reserved {reserved:.2f} GB')
    return stats


def safe_delete(*objs) -> None:
    """Delete references and immediately trigger memory reclaim."""
    for obj in objs:
        try:
            del obj
        except Exception:
            pass
    clear_gpu_memory()


def monitor_vram_guard(min_free_gb: float = 1.5) -> bool:
    """
    Check free VRAM. Returns True if safe to continue.

    Raises a warning and returns False if below min_free_gb threshold.
    """
    if not torch.cuda.is_available():
        return True
    total = torch.cuda.get_device_properties(0).total_memory
    alloc = torch.cuda.memory_allocated(0)
    free_gb = (total - alloc) / 1e9
    if free_gb < min_free_gb:
        print(f'⚠️  VRAM low: {free_gb:.2f} GB free (threshold {min_free_gb} GB). '
              f'Consider reducing resolution or frame count.')
        return False
    return True


# ── Load nodes (deferred — called inside generation) ───────────────────────
_NODES_LOADED = False
NODE_CLASS_MAPPINGS = None

def ensure_nodes_loaded() -> None:
    """Idempotently load all ComfyUI nodes."""
    global _NODES_LOADED, NODE_CLASS_MAPPINGS
    if _NODES_LOADED:
        return
    print('🔌 Loading ComfyUI nodes...')
    os.chdir(str(COMFYUI_DIR))
    import_custom_nodes()
    from nodes import NODE_CLASS_MAPPINGS as _NCM
    NODE_CLASS_MAPPINGS = _NCM
    _NODES_LOADED = True
    print(f'  ✅ {len(NODE_CLASS_MAPPINGS)} nodes available.')


print('✅ Workflow parser and helpers defined.')
print('   (Nodes will be loaded lazily on first generation)')
print('\n✅ Section 6 complete.')

# ⚙️ Section 7 · Runtime Configuration

In [ ]:
# @title ⚙️ 7. Runtime Configuration { display-mode: "form" }
# @markdown Core generation parameters. Hardware limits are enforced automatically.

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  GENERATION SETTINGS                                                   ║
# ╚══════════════════════════════════════════════════════════════════════════╝

# ── Resolution (max 480p for T4) ───────────────────────────────────────────
OUTPUT_WIDTH  = 832  # @param ["480", "640", "704", "768", "832"] {type:"raw"}
OUTPUT_HEIGHT = 480  # @param ["272", "360", "480"] {type:"raw"}

# ── Duration & frame rate ──────────────────────────────────────────────────
# Max 4 s × 25 fps = 100 frames. Use 97 for clean 8n+1 alignment.
DURATION_SECONDS = 4.0   # @param {type:"number"}
FPS              = 25    # @param ["24", "25", "30"] {type:"raw"}

# ── Seed ──────────────────────────────────────────────────────────────────
SEED             = 42    # @param {type:"integer"}
RANDOMIZE_SEED   = False # @param {type:"boolean"}

# ── Sampler (matches reference workflow) ──────────────────────────────────
#  Pass-1 sampler  : euler   (coarse denoising)
#  Pass-2 sampler  : gradient_estimation  (refinement)
SAMPLER_PASS1    = 'euler'                # @param {type:"string"}
SAMPLER_PASS2    = 'gradient_estimation'  # @param {type:"string"}

# ── Manual sigmas (distilled schedule from reference workflow) ─────────────
SIGMAS_PASS1 = '1., 0.99375, 0.9875, 0.98125, 0.975, 0.909375, 0.725, 0.421875, 0.0'
SIGMAS_PASS2 = '0.909375, 0.725, 0.421875, 0.0'

# ── CFG scale (distilled model uses cfg=1) ─────────────────────────────────
CFG_SCALE        = 1.0   # @param {type:"number"}

# ── Image preprocessing ────────────────────────────────────────────────────
IMG_COMPRESSION  = 33    # @param {type:"integer"}
LONGER_EDGE      = 848   # @param {type:"integer"}
IMAGE_STRENGTH   = 1.0   # @param {type:"number"}

# ── Output ─────────────────────────────────────────────────────────────────
OUTPUT_FOLDER    = '/content/outputs'   # @param {type:"string"}
OUTPUT_PREFIX    = 'LTX2_SVI'          # @param {type:"string"}
SAVE_METADATA    = True   # @param {type:"boolean"}
SAVE_FIRST_FRAME = True   # @param {type:"boolean"}
SAVE_PREVIEW_GIF = False  # @param {type:"boolean"}

# ══════════════════════════════════════════════════════════════════════════

# ── Hardware safety clamps ─────────────────────────────────────────────────
MAX_WIDTH_T4  = 832
MAX_HEIGHT_T4 = 480
MAX_DURATION  = 4.0
MAX_FPS       = 30

OUTPUT_WIDTH    = min(int(OUTPUT_WIDTH),    MAX_WIDTH_T4)
OUTPUT_HEIGHT   = min(int(OUTPUT_HEIGHT),   MAX_HEIGHT_T4)
DURATION_SECONDS = min(float(DURATION_SECONDS), MAX_DURATION)
FPS             = min(int(FPS), MAX_FPS)

# LTX-2 requires frame count = 8n+1
raw_frames = int(DURATION_SECONDS * FPS)
FRAME_COUNT = ((raw_frames - 1) // 8) * 8 + 1  # round DOWN to 8n+1
FRAME_COUNT = max(FRAME_COUNT, 9)               # minimum 9 frames

# Latent spatial dims = pixel dim // 2 (after scale-by-0.5)
LATENT_W = OUTPUT_WIDTH  // 2
LATENT_H = OUTPUT_HEIGHT // 2

if RANDOMIZE_SEED:
    import random
    SEED = random.randint(0, 2**32 - 1)

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

print('⚙️  Runtime Configuration')
print('=' * 50)
print(f'  Resolution  : {OUTPUT_WIDTH} × {OUTPUT_HEIGHT}')
print(f'  Frames      : {FRAME_COUNT}  ({FRAME_COUNT/FPS:.2f}s @ {FPS} fps)')
print(f'  Seed        : {SEED}')
print(f'  CFG         : {CFG_SCALE}')
print(f'  Sampler P1  : {SAMPLER_PASS1}')
print(f'  Sampler P2  : {SAMPLER_PASS2}')
print(f'  Sigmas P1   : {SIGMAS_PASS1}')
print(f'  Sigmas P2   : {SIGMAS_PASS2}')
print(f'  Latent dims : {LATENT_W} × {LATENT_H}')
print(f'  Output dir  : {OUTPUT_FOLDER}')
print('\n✅ Section 7 complete.')

# 🎬 Section 8 · Story Scene Configuration

In [ ]:
# @title 🎬 8. Story Scene Configuration { display-mode: "form" }
# @markdown Define your story scenes. Each scene maps directly to one generated clip.
# @markdown Character references propagate automatically from the character database.

from dataclasses import dataclass, field
from typing import List, Optional


@dataclass
class Scene:
    """
    A single story beat / generated clip.

    Attributes mirror the Director 2.0 workflow segment structure:
    global_prompt, per-segment prompt, image reference, duration, etc.
    """
    scene_id:          int
    prompt:            str
    negative_prompt:   str            = ''
    image_path:        Optional[str]  = None
    # Cinematic language
    camera_movement:   str            = 'slow push-in'
    lens:              str            = '35mm'
    lighting:          str            = 'natural cinematic'
    location:          str            = ''
    emotion:           str            = 'neutral'
    # Narrative
    dialogue:          str            = ''
    duration_seconds:  float          = 4.0
    transition:        str            = 'cut'
    # Sound design (metadata only)
    music_notes:       str            = ''
    sound_effects:     str            = ''
    # Character IDs to inject (from CharacterDatabase)
    character_ids:     List[str]      = field(default_factory=list)
    # Generation overrides (inherit global config if None)
    seed:              Optional[int]  = None
    width:             Optional[int]  = None
    height:            Optional[int]  = None
    fps:               Optional[int]  = None


# ── Shared negative prompt ─────────────────────────────────────────────────
GLOBAL_NEGATIVE_PROMPT = (
    'worst quality, inconsistent motion, blurry, jittery, distorted, '
    'watermark, text overlay, duplicate, blurry eyes, morphing face, '
    'deformed hands, extra fingers, extra limbs, bad anatomy, '
    'flickering, color bleeding, overexposed, underexposed'
)

# ── Global director prompt (character-consistent performance style) ─────────
GLOBAL_DIRECTOR_PROMPT = (
    'Cinematic photorealistic video. Preserve character identity, '
    'facial structure, hairstyle, skin tone and clothing exactly. '
    'Smooth camera motion, natural lighting, stable motion. '
    'High quality, premium cinematography, no jitter, no flickering.'
)

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  STORY SCENES — Edit freely                                            ║
# ╚══════════════════════════════════════════════════════════════════════════╝

STORY_SCENES: List[Scene] = [
    Scene(
        scene_id        = 1,
        prompt          = 'A confident young woman walks toward the camera through a sun-lit forest path, dappled light, 4K cinematic',
        camera_movement = 'slow push-in',
        lens            = '35mm',
        lighting        = 'golden hour, dappled sunlight',
        location        = 'forest path',
        emotion         = 'confident, calm',
        duration_seconds= 4.0,
        character_ids   = ['char_hero'],
    ),
    Scene(
        scene_id        = 2,
        prompt          = 'She pauses, looks up at the sky, a gentle breeze moves her hair, cinematic close-up',
        camera_movement = 'slow orbit',
        lens            = '85mm portrait',
        lighting        = 'soft overcast sky',
        location        = 'forest clearing',
        emotion         = 'wonder, contemplation',
        duration_seconds= 4.0,
        character_ids   = ['char_hero'],
    ),
]

# ── Active scene selection ─────────────────────────────────────────────────
# Set to None to use STORY_SCENES, or an integer index (0-based) for one scene.
ACTIVE_SCENE_INDEX = 0  # @param {type:"integer"}

print('🎬 Story Configuration')
print('=' * 50)
for s in STORY_SCENES:
    print(f'  Scene {s.scene_id}: {s.prompt[:60]}...')
    print(f'           Camera: {s.camera_movement} | Lens: {s.lens} | {s.duration_seconds}s')
print(f'\n  Active scene index: {ACTIVE_SCENE_INDEX}')
print('\n✅ Section 8 complete.')

# 👤 Section 9 · Character Database

In [ ]:
# @title 👤 9. Character Database { display-mode: "form" }
# @markdown Persistent character identities reused across all scenes.
# @markdown Characters are compiled into prompt tokens automatically.

@dataclass
class Character:
    """
    A reusable character identity.

    Every field contributes to the auto-compiled character description
    that gets injected into each scene prompt. Using consistent character
    tokens across generations reduces identity drift.
    """
    # ── Identity ──────────────────────────────────────────────────────────
    char_id:          str
    name:             str
    age:              str                   = '30s'
    gender:           str                   = 'woman'
    # ── Physical appearance ────────────────────────────────────────────────
    appearance:       str                   = 'athletic build'
    hairstyle:        str                   = 'long dark hair'
    face:             str                   = 'oval face, warm brown eyes'
    body_proportions: str                   = 'average height'
    skin_tone:        str                   = 'warm olive skin'
    # ── Styling ────────────────────────────────────────────────────────────
    clothing:         str                   = 'casual modern outfit'
    accessories:      str                   = ''
    # ── Personality & performance ──────────────────────────────────────────
    personality:      str                   = 'confident, expressive'
    expressions:      str                   = 'natural, emotive'
    pose_references:  List[str]             = field(default_factory=list)
    motion_style:     str                   = 'natural, smooth'
    # ── Technical ─────────────────────────────────────────────────────────
    seed:             Optional[int]         = None
    reference_image:  Optional[str]         = None   # path to reference image
    lora_references:  List[str]             = field(default_factory=list)
    negative_additions: str                 = ''
    # ── Camera & lighting preferences ─────────────────────────────────────
    lighting_pref:    str                   = 'soft cinematic lighting'
    camera_pref:      str                   = 'medium shot, slight low angle'

    def to_prompt_tokens(self) -> str:
        """
        Compile this character into a dense prompt description string.

        The string is injected at the start of every scene prompt to
        anchor character identity across multi-scene generation.
        """
        tokens = [
            f'{self.name}, {self.age} {self.gender}',
            self.appearance,
            self.hairstyle,
            self.face,
            self.skin_tone,
            self.body_proportions,
            self.clothing,
        ]
        if self.accessories:
            tokens.append(self.accessories)
        # Filter empty strings
        return ', '.join(t for t in tokens if t.strip())


class CharacterDatabase:
    """
    In-memory registry for all characters.

    Characters are registered once and automatically injected into
    any scene that references their char_id.
    """

    def __init__(self):
        self._chars: Dict[str, Character] = {}

    def register(self, character: Character) -> None:
        """Add or update a character."""
        self._chars[character.char_id] = character
        print(f'  ✅ Registered: {character.name} (id={character.char_id})')

    def get(self, char_id: str) -> Optional[Character]:
        """Retrieve a character by ID."""
        return self._chars.get(char_id)

    def get_prompt_tokens(self, char_ids: List[str]) -> str:
        """
        Return merged prompt tokens for a list of character IDs.

        Multiple characters are separated by ' and ' for readability.
        """
        parts = []
        for cid in char_ids:
            char = self.get(cid)
            if char:
                parts.append(char.to_prompt_tokens())
            else:
                print(f'  ⚠️  Unknown character id: {cid}')
        return ' and '.join(parts)

    def get_reference_image(self, char_ids: List[str]) -> Optional[str]:
        """Return the first available reference image from the character list."""
        for cid in char_ids:
            char = self.get(cid)
            if char and char.reference_image and Path(char.reference_image).exists():
                return char.reference_image
        return None

    def list(self) -> None:
        """Print all registered characters."""
        print(f'  Characters in database ({len(self._chars)}):')
        for cid, char in self._chars.items():
            print(f'    [{cid}] {char.name} — {char.age} {char.gender}')


# ── Initialise database ────────────────────────────────────────────────────
CHARACTER_DB = CharacterDatabase()

# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  REGISTER YOUR CHARACTERS — Edit freely                                ║
# ╚══════════════════════════════════════════════════════════════════════════╝

CHARACTER_DB.register(Character(
    char_id          = 'char_hero',
    name             = 'Maya',
    age              = 'late 20s',
    gender           = 'woman',
    appearance       = 'athletic slender build',
    hairstyle        = 'long wavy dark brown hair',
    face             = 'high cheekbones, expressive hazel eyes, full lips',
    skin_tone        = 'warm golden tan skin',
    body_proportions = 'tall, 5ft8',
    clothing         = 'fitted white linen shirt, light beige trousers',
    accessories      = 'small gold hoop earrings',
    personality      = 'confident, curious, warm',
    motion_style     = 'graceful, purposeful',
    lighting_pref    = 'golden hour side lighting',
    camera_pref      = 'medium full shot, slightly low angle',
    # reference_image = '/content/ComfyUI/input/maya_reference.jpg',  # optional
))

print('\n👤 Character Database')
CHARACTER_DB.list()
print('\n✅ Section 9 complete.')

# ✍️ Section 10 · Automatic Prompt Compiler

In [ ]:
# @title ✍️ 10. Automatic Prompt Compiler { display-mode: "form" }
# @markdown Merges character tokens, scene description, camera language,
# @markdown lighting, composition, motion, environment, style, and quality tokens.

# ── Quality tokens injected into every positive prompt ─────────────────────
QUALITY_TOKENS = (
    'best quality, 4K ultra HD, photorealistic, cinematic, '
    'sharp focus, professional cinematography, premium production'
)

# ── Style tokens ───────────────────────────────────────────────────────────
STYLE_TOKENS = 'cinematic color grading, film grain, natural motion blur'


def compile_prompt(
    scene: 'Scene',
    character_db: 'CharacterDatabase',
    global_director_prompt: str = '',
    extra_tokens: str = '',
) -> str:
    """
    Build the final positive prompt for a scene.

    Compilation order:
      1. Character identity tokens (from CharacterDatabase)
      2. Scene description (scene.prompt)
      3. Camera language
      4. Lighting
      5. Location / environment
      6. Emotion / performance
      7. Dialogue (if present)
      8. Motion style
      9. Style tokens
     10. Quality tokens
     11. Global director prompt
     12. Extra tokens

    Args:
        scene                  : Scene dataclass instance.
        character_db           : CharacterDatabase for token lookup.
        global_director_prompt : Persistent director-level prompt.
        extra_tokens           : Any additional user tokens.

    Returns:
        A single comma-separated prompt string.
    """
    parts = []

    # 1. Character tokens
    if scene.character_ids:
        char_tokens = character_db.get_prompt_tokens(scene.character_ids)
        if char_tokens:
            parts.append(char_tokens)

    # 2. Scene description
    if scene.prompt:
        parts.append(scene.prompt)

    # 3. Camera
    if scene.camera_movement:
        parts.append(f'{scene.camera_movement} camera movement')
    if scene.lens:
        parts.append(f'{scene.lens} lens')

    # 4. Lighting
    if scene.lighting:
        parts.append(scene.lighting)

    # 5. Location
    if scene.location:
        parts.append(f'location: {scene.location}')

    # 6. Emotion
    if scene.emotion:
        parts.append(f'emotion: {scene.emotion}')

    # 7. Dialogue
    if scene.dialogue:
        parts.append(f'spoken dialogue: "{scene.dialogue}"')

    # 8. Motion style (from character if available)
    char = None
    if scene.character_ids:
        char = character_db.get(scene.character_ids[0])
    if char and char.motion_style:
        parts.append(f'{char.motion_style} movement')

    # 9–10. Style + quality
    parts.append(STYLE_TOKENS)
    parts.append(QUALITY_TOKENS)

    # 11. Global director prompt
    if global_director_prompt:
        parts.append(global_director_prompt)

    # 12. Extra
    if extra_tokens:
        parts.append(extra_tokens)

    return ', '.join(p.strip().rstrip(',') for p in parts if p.strip())


def compile_negative_prompt(
    scene: 'Scene',
    character_db: 'CharacterDatabase',
    global_negative: str = GLOBAL_NEGATIVE_PROMPT,
) -> str:
    """
    Merge scene negative prompt with character additions and global negative.

    Returns a single comma-separated negative prompt string.
    """
    parts = [global_negative]

    if scene.negative_prompt:
        parts.append(scene.negative_prompt)

    if scene.character_ids:
        for cid in scene.character_ids:
            char = character_db.get(cid)
            if char and char.negative_additions:
                parts.append(char.negative_additions)

    return ', '.join(p.strip() for p in parts if p.strip())


# ── Preview compiled prompts for active scene ──────────────────────────────
if STORY_SCENES:
    _s = STORY_SCENES[ACTIVE_SCENE_INDEX]
    _pos = compile_prompt(_s, CHARACTER_DB, GLOBAL_DIRECTOR_PROMPT)
    _neg = compile_negative_prompt(_s, CHARACTER_DB)

    print('✍️  Compiled Prompts (Scene', _s.scene_id, ')')
    print('=' * 60)
    print('POSITIVE:')
    print(f'  {_pos[:300]}...' if len(_pos) > 300 else f'  {_pos}')
    print('\nNEGATIVE:')
    print(f'  {_neg[:200]}...' if len(_neg) > 200 else f'  {_neg}')

print('\n✅ Section 10 complete.')

# 🎥 Section 11 · Director Guidance System

In [ ]:
# @title 🎥 11. Director Guidance System { display-mode: "form" }
# @markdown Implements the LTXDirectorGuide and LTXDirectorCropGuides nodes
# @markdown from the Director 2.0 workflow. These wrap conditioning and
# @markdown latents through guide_data for multi-pass refinement.


class DirectorGuideConfig:
    """
    Configuration mirror of the LTXDirectorGuide node.

    Parameters match the widget_values from LTX-2.3_Director_2.0 workflow:
      - guide_mode          : 'None' (standard) or spatial mode variant
      - guide_strength      : Pass-1 guide strength (1.0 = full)
      - guide_strength_p2   : Pass-2 guide strength (0.5 = half-strength refine)
      - upsample_method     : Latent upsample method ('bicubic')
      - upsample_scale      : Spatial upscale factor (1 = no extra scale)
      - crop_position       : Crop anchor ('center')
      - use_director_guide  : Enable/disable director guidance
      - use_motion_guide    : Enable motion guide data
      - guide_width         : Guide tile width (256)
      - guide_height        : Guide tile height (64)
      - debug               : Print internal director stats
    """

    def __init__(
        self,
        guide_mode:         str   = 'None',
        guide_strength:     float = 1.0,
        guide_strength_p2:  float = 0.5,
        upsample_method:    str   = 'bicubic',
        upsample_scale:     int   = 1,
        crop_position:      str   = 'center',
        use_director_guide: bool  = True,
        use_motion_guide:   bool  = False,
        guide_width:        int   = 256,
        guide_height:       int   = 64,
        debug:              bool  = False,
    ):
        self.guide_mode         = guide_mode
        self.guide_strength     = guide_strength
        self.guide_strength_p2  = guide_strength_p2
        self.upsample_method    = upsample_method
        self.upsample_scale     = upsample_scale
        self.crop_position      = crop_position
        self.use_director_guide = use_director_guide
        self.use_motion_guide   = use_motion_guide
        self.guide_width        = guide_width
        self.guide_height       = guide_height
        self.debug              = debug


class DirectorGuidePipeline:
    """
    Python wrapper for the LTXDirectorGuide + LTXDirectorCropGuides nodes.

    In the Director 2.0 workflow these nodes sit between conditioning
    and the CFGGuider at each pass, adding guide_data (per-frame spatial
    anchoring from the input image segments) to the conditioning.

    When the LTXDirectorGuide custom node is NOT installed (e.g., running
    the lightweight distilled-only pipeline), this class transparently
    passes conditioning through unchanged, preserving full compatibility.
    """

    def __init__(self, config: DirectorGuideConfig = None):
        self.config = config or DirectorGuideConfig()
        self._node_available = None  # lazily detected

    def _check_node_available(self) -> bool:
        """Check whether the LTXDirectorGuide node is loaded."""
        if self._node_available is None:
            self._node_available = (
                NODE_CLASS_MAPPINGS is not None and
                'LTXDirectorGuide' in NODE_CLASS_MAPPINGS
            )
        return self._node_available

    def apply_guide_pass1(
        self,
        positive,
        negative,
        vae,
        latent,
        guide_data=None,
        motion_guide_data=None,
        model=None,
    ):
        """
        Apply director guidance for Pass-1 (coarse sampling).

        Maps to LTXDirectorGuide node with guide_strength=1.0 (workflow node 132).
        Returns (positive, negative, latent, model) — same output shape as the node.
        """
        if not self._check_node_available() or guide_data is None:
            # Passthrough: no director node available
            return positive, negative, latent, model

        node = NODE_CLASS_MAPPINGS['LTXDirectorGuide']()
        result = node.EXECUTE_NORMALIZED(
            guide_mode          = self.config.guide_mode,
            guide_strength      = self.config.guide_strength,
            guide_strength_p2   = self.config.guide_strength_p2,
            upsample_method     = self.config.upsample_method,
            upsample_scale      = self.config.upsample_scale,
            crop_position       = self.config.crop_position,
            use_director_guide  = self.config.use_director_guide,
            use_motion_guide    = self.config.use_motion_guide,
            guide_width         = self.config.guide_width,
            guide_height        = self.config.guide_height,
            positive            = positive,
            negative            = negative,
            vae                 = vae,
            latent              = latent,
            guide_data          = guide_data,
            motion_guide_data   = motion_guide_data,
            model               = model,
        )
        pos_out   = get_value_at_index(result, 0)
        neg_out   = get_value_at_index(result, 1)
        lat_out   = get_value_at_index(result, 2)
        model_out = get_value_at_index(result, 3)
        return pos_out, neg_out, lat_out, model_out

    def apply_crop_guides(
        self,
        positive,
        negative,
        latent,
    ):
        """
        Apply LTXDirectorCropGuides (or LTXVCropGuides for base pipeline).

        Maps to the LTXDirectorCropGuides node (workflow nodes 54/55)
        which crops guide annotations from the conditioning after Pass-1
        to prepare the latent for upscaling + Pass-2.

        Falls back to LTXVCropGuides if the Director node is absent.
        """
        if NODE_CLASS_MAPPINGS is None:
            return positive, negative, latent

        # Prefer Director variant, fall back to standard crop guides
        node_name = (
            'LTXDirectorCropGuides' if 'LTXDirectorCropGuides' in NODE_CLASS_MAPPINGS
            else 'LTXVCropGuides'
        )

        if node_name not in NODE_CLASS_MAPPINGS:
            return positive, negative, latent

        node   = NODE_CLASS_MAPPINGS[node_name]()
        result = node.EXECUTE_NORMALIZED(
            positive = positive,
            negative = negative,
            latent   = latent,
        )
        return (
            get_value_at_index(result, 0),
            get_value_at_index(result, 1),
            get_value_at_index(result, 2),
        )


# ── Global director pipeline instance ─────────────────────────────────────
DIRECTOR = DirectorGuidePipeline(
    DirectorGuideConfig(
        guide_strength    = 1.0,
        guide_strength_p2 = 0.5,
        use_director_guide= True,
        use_motion_guide  = False,
    )
)

print('🎥 Director Guidance System initialized.')
print(f'   Guide strength P1 : {DIRECTOR.config.guide_strength}')
print(f'   Guide strength P2 : {DIRECTOR.config.guide_strength_p2}')
print(f'   Upsample method   : {DIRECTOR.config.upsample_method}')
print('\n✅ Section 11 complete.')

# 🧊 Section 12 · Latent Generation

In [ ]:
# @title 🧊 12. Latent Generation { display-mode: "form" }
# @markdown Image preprocessing → VAE encode → EmptyLTXVLatentVideo →
# @markdown LTXVImgToVideoInplace → audio latent → LTXVConcatAVLatent.
# @markdown Mirrors the full latent setup pipeline from the reference workflow.


def build_image_latent(
    image_path:       Optional[str],
    width:            int,
    height:           int,
    frame_count:      int,
    fps:              int,
    img_compression:  int,
    longer_edge:      int,
    image_strength:   float,
    vae_model_name:   str,
    audio_vae_name:   str,
    latent_w:         int,
    latent_h:         int,
) -> tuple:
    """
    Build the combined audio-video latent for the LTX-2 pipeline.

    Pipeline (faithful to reference ltx2_ti2v_distilled-.py):
      1. LoadImage / synthetic noise tensor (T2V)
      2. ResizeImageMaskNode  → target (W×H)
      3. ResizeImagesByLongerEdge → longer_edge=848 (preprocessing)
      4. LTXVPreprocess  (img_compression=33)
      5. EmptyLTXVLatentVideo  (latent_w×latent_h, frame_count)
      6. VAELoader  → LTXVImgToVideoInplace  (or bypass for T2V)
      7. VAELoaderKJ (audio) → LTXVEmptyLatentAudio
      8. LTXVConcatAVLatent  → combined latent

    Args:
        image_path      : Path to input image, or None for T2V.
        width/height    : Target video resolution.
        frame_count     : Number of frames (8n+1 aligned).
        fps             : Frame rate.
        img_compression : LTXVPreprocess compression factor.
        longer_edge     : ResizeImagesByLongerEdge target.
        image_strength  : img-to-vid conditioning strength.
        vae_model_name  : Video VAE filename.
        audio_vae_name  : Audio VAE filename.
        latent_w/h      : Pre-computed latent spatial dims.

    Returns:
        Tuple: (av_concat_latent, preprocessed_image, image_bypass, image_strength)
    """
    ncm = NODE_CLASS_MAPPINGS

    # ── 1. Image or synthetic noise ────────────────────────────────────────
    image_bypass = False
    if image_path is not None and Path(image_path).exists():
        loadimage = ncm['LoadImage']()
        loaded_image = loadimage.load_image(image=image_path)
        img_strength  = image_strength
        image_bypass  = False
        print(f'  📷 Loaded image: {image_path}')
    else:
        # T2V: fill with mid-gray noise
        noise = torch.full((1, height, width, 3), 0.5)
        loaded_image = (noise, None)
        img_strength  = 0.0
        image_bypass  = True
        print('  🎨 Text-to-Video mode (no input image)')

    # ── 2. Resize to target W×H ────────────────────────────────────────────
    resize_node = ncm['ResizeImageMaskNode']()
    resized = resize_node.EXECUTE_NORMALIZED(
        input        = get_value_at_index(loaded_image, 0),
        scale_method = 'lanczos',
        resize_type  = {
            'resize_type': 'scale dimensions',
            'width':  width,
            'height': height,
            'crop':   'center',
        }
    )

    # ── 3. Resize by longer edge (preprocessing resolution) ────────────────
    longer_edge_node = ncm['ResizeImagesByLongerEdge']()
    preprocessed_long = longer_edge_node.EXECUTE_NORMALIZED(
        longer_edge = longer_edge,
        images      = get_value_at_index(resized, 0),
    )

    # ── 4. LTXVPreprocess ──────────────────────────────────────────────────
    preprocess_node = ncm['LTXVPreprocess']()
    preprocessed = preprocess_node.EXECUTE_NORMALIZED(
        img_compression = img_compression,
        image           = get_value_at_index(preprocessed_long, 0),
    )

    # ── 5. EmptyLTXVLatentVideo ────────────────────────────────────────────
    empty_latent_node = ncm['EmptyLTXVLatentVideo']()
    empty_latent = empty_latent_node.EXECUTE_NORMALIZED(
        width      = latent_w,
        height     = latent_h,
        length     = frame_count,
        batch_size = 1,
    )

    # ── 6. VAELoader + LTXVImgToVideoInplace ──────────────────────────────
    vae_loader = ncm['VAELoader']()
    vae        = vae_loader.load_vae(vae_name=vae_model_name)

    img2vid_node = ncm['LTXVImgToVideoInplace']()
    img2vid = img2vid_node.EXECUTE_NORMALIZED(
        strength = img_strength,
        bypass   = image_bypass,
        vae      = get_value_at_index(vae, 0),
        image    = get_value_at_index(preprocessed, 0),
        latent   = get_value_at_index(empty_latent, 0),
    )

    # Free video VAE immediately after encode
    del vae
    clear_gpu_memory()
    print('  ✅ Video latent encoded.')

    # ── 7. Audio VAE + EmptyLatentAudio ────────────────────────────────────
    audio_vae = load_audio_vae_compat(audio_vae_name)

    empty_audio_node = ncm['LTXVEmptyLatentAudio']()
    empty_audio = empty_audio_node.EXECUTE_NORMALIZED(
        frames_number = frame_count,
        frame_rate    = fps,
        batch_size    = 1,
        audio_vae     = get_value_at_index(audio_vae, 0),
    )

    # ── 8. Concatenate AV latent ────────────────────────────────────────────
    concat_node = ncm['LTXVConcatAVLatent']()

    # Use img2vid latent for I2V, empty latent for T2V (matches reference logic)
    video_latent = (
        get_value_at_index(img2vid, 0)
        if not image_bypass
        else get_value_at_index(empty_latent, 0)
    )

    av_latent = concat_node.EXECUTE_NORMALIZED(
        video_latent = video_latent,
        audio_latent = get_value_at_index(empty_audio, 0),
    )

    print('  ✅ Audio+Video latent concatenated.')
    log_memory('latent_build')

    return (
        av_latent,
        preprocessed,        # preprocessed image (for Pass-2 inplace)
        image_bypass,
        img_strength,
        audio_vae,           # kept alive for audio decode later
        empty_latent,        # kept for Pass-2 upsampler input reference
    )


print('🧊 Latent generation functions defined.')
print('\n✅ Section 12 complete.')

# 🎲 Section 13 · Dual-Pass Sampler Execution

In [ ]:
# @title 🎲 13. Dual-Pass Sampler Execution { display-mode: "form" }
# @markdown Implements the full dual-pass distilled sampling pipeline:
# @markdown Pass-1 (euler, ManualSigmas) → CropGuides → LatentUpsampler → Pass-2 (gradient_estimation).
# @markdown Faithfully mirrors every node connection in the reference workflow.


def run_text_encoding(
    prompt:       str,
    neg_prompt:   str,
    fps:          int,
) -> tuple:
    """
    DualCLIPLoader → CLIPTextEncode → ConditioningZeroOut → LTXVConditioning.

    This exactly mirrors the text conditioning pipeline from the reference:
      DualCLIPLoader(gemma fp4 + embeddings_connector) →
      CLIPTextEncode(prompt) →
      ConditioningZeroOut(→ negative) →
      LTXVConditioning(frame_rate=fps)

    Returns:
        (positive_conditioning, negative_conditioning)
    """
    ncm = NODE_CLASS_MAPPINGS

    print('  📝 Loading text encoders...')
    dual_clip = ncm['DualCLIPLoader']()
    try:
        clip = dual_clip.load_clip(
            clip_name1 = TEXT_ENC1_MODEL,
            clip_name2 = TEXT_ENC2_MODEL,
            type       = 'ltxv',
            device     = 'default',
        )
    except Exception as e:
        print(f'  ⚠️  Primary CLIP load failed ({e}), retrying...')
        clip = dual_clip.load_clip(
            clip_name1 = TEXT_ENC1_MODEL,
            clip_name2 = TEXT_ENC2_MODEL,
            type       = 'ltxv',
            device     = 'cpu',
        )

    encode_node = ncm['CLIPTextEncode']()
    positive_raw = encode_node.encode(
        text = prompt,
        clip = get_value_at_index(clip, 0),
    )

    # Free CLIP immediately (saves ~3 GB)
    del clip
    clear_gpu_memory()
    print('  ✅ Text encoded. CLIP freed.')

    # Negative = zero-out of positive embedding
    zero_out   = ncm['ConditioningZeroOut']()
    negative_raw = zero_out.zero_out(
        conditioning = get_value_at_index(positive_raw, 0)
    )

    # LTXVConditioning injects frame_rate metadata into both
    ltxv_cond = ncm['LTXVConditioning']()
    conditioning = ltxv_cond.EXECUTE_NORMALIZED(
        frame_rate = fps,
        positive   = get_value_at_index(positive_raw, 0),
        negative   = get_value_at_index(negative_raw, 0),
    )

    pos_cond = get_value_at_index(conditioning, 0)
    neg_cond = get_value_at_index(conditioning, 1)
    print(f'  ✅ LTXVConditioning applied (frame_rate={fps}).')

    return pos_cond, neg_cond


def load_dit_model(lora_1=None, lora_2=None, lora_3=None,
                   lora_s1=1.0, lora_s2=1.0, lora_s3=1.0):
    """
    UnetLoaderGGUF → LoraLoaderModelOnly (×3 if enabled).

    Returns the model tensor (unwrapped from the loader tuple).
    """
    from nodes import LoraLoaderModelOnly
    ncm = NODE_CLASS_MAPPINGS

    print('  🧠 Loading DiT (GGUF)...')
    gguf_loader = ncm['UnetLoaderGGUF']()
    model_tuple = gguf_loader.load_unet(unet_name=DIT_MODEL)
    model = get_value_at_index(model_tuple, 0)

    lora_loader = LoraLoaderModelOnly()
    if lora_1:
        print(f'  🔗 Applying LoRA 1: {lora_1} (str={lora_s1})')
        model = lora_loader.load_lora_model_only(model, lora_1, lora_s1)[0]
    if lora_2:
        print(f'  🔗 Applying LoRA 2: {lora_2} (str={lora_s2})')
        model = lora_loader.load_lora_model_only(model, lora_2, lora_s2)[0]
    if lora_3:
        print(f'  🔗 Applying LoRA 3: {lora_3} (str={lora_s3})')
        model = lora_loader.load_lora_model_only(model, lora_3, lora_s3)[0]

    print('  ✅ DiT loaded.')
    log_memory('dit_loaded')
    return model


def run_dual_pass_sampling(
    pos_cond,
    neg_cond,
    av_latent,
    preprocessed_image,
    audio_vae,
    image_bypass:   bool,
    image_strength: float,
    seed:           int,
    cfg:            float,
    sampler_p1:     str,
    sampler_p2:     str,
    sigmas_p1:      str,
    sigmas_p2:      str,
    vae_model_name: str,
    upscaler_name:  str,
    fps:            int,
    frame_count:    int,
    latent_w:       int,
    latent_h:       int,
    model,
) -> tuple:
    """
    Execute the dual-pass LTX-2 distilled sampling pipeline.

    === Pass 1 (coarse denoising) ===
      KSamplerSelect(euler) + ManualSigmas('1., 0.99375…0.0') +
      RandomNoise(seed) + CFGGuider(cfg=1) →
      SamplerCustomAdvanced → LTXVSeparateAVLatent

    === Between passes ===
      LTXVCropGuides → LTXVLatentUpsampler(x2) → LTXVImgToVideoInplace

    === Pass 2 (refinement) ===
      KSamplerSelect(gradient_estimation) + ManualSigmas('0.909375…0.0') +
      RandomNoise(seed=0) + CFGGuider(cfg=1) →
      SamplerCustomAdvanced → LTXVSeparateAVLatent

    Returns:
        (video_latent_final, audio_latent_final)
    """
    ncm = NODE_CLASS_MAPPINGS

    # ── Instantiate stateless node wrappers ────────────────────────────────
    ksampler_select = ncm['KSamplerSelect']()
    manual_sigmas   = ncm['ManualSigmas']()
    random_noise    = ncm['RandomNoise']()
    cfg_guider      = ncm['CFGGuider']()
    sampler_node    = ncm['SamplerCustomAdvanced']()
    sep_av          = ncm['LTXVSeparateAVLatent']()
    crop_guides     = ncm['LTXVCropGuides']()
    upsampler       = ncm['LTXVLatentUpsampler']()
    concat_av       = ncm['LTXVConcatAVLatent']()
    img2vid         = ncm['LTXVImgToVideoInplace']()
    vae_loader      = ncm['VAELoader']()
    upscale_loader  = ncm['LatentUpscaleModelLoader']()

    # ── Sampler selects ────────────────────────────────────────────────────
    sampler_p1_obj = ksampler_select.EXECUTE_NORMALIZED(sampler_name=sampler_p1)
    sampler_p2_obj = ksampler_select.EXECUTE_NORMALIZED(sampler_name=sampler_p2)

    # ── Sigma schedules ────────────────────────────────────────────────────
    sigmas_p1_obj = manual_sigmas.EXECUTE_NORMALIZED(sigmas=sigmas_p1)
    sigmas_p2_obj = manual_sigmas.EXECUTE_NORMALIZED(sigmas=sigmas_p2)

    # ── Noise generators ───────────────────────────────────────────────────
    # Pass-1: user seed  |  Pass-2: seed=0 (fixed per reference workflow)
    noise_p1 = random_noise.EXECUTE_NORMALIZED(noise_seed=seed)
    noise_p2 = random_noise.EXECUTE_NORMALIZED(noise_seed=0)

    # ═══════════════════════════════════════════════════════════════════════
    #  PASS 1  —  Coarse denoising
    # ═══════════════════════════════════════════════════════════════════════
    print('  🎲 Pass-1 sampling (euler)...')

    guider_p1 = cfg_guider.EXECUTE_NORMALIZED(
        cfg      = cfg,
        model    = model,
        positive = pos_cond,
        negative = neg_cond,
    )

    monitor_vram_guard(min_free_gb=1.0)

    sample_p1 = sampler_node.EXECUTE_NORMALIZED(
        noise        = get_value_at_index(noise_p1, 0),
        guider       = get_value_at_index(guider_p1, 0),
        sampler      = get_value_at_index(sampler_p1_obj, 0),
        sigmas       = get_value_at_index(sigmas_p1_obj, 0),
        latent_image = get_value_at_index(av_latent, 0),
    )

    del guider_p1
    clear_gpu_memory()
    print('  ✅ Pass-1 complete.')
    log_memory('pass1_done')

    # Separate video/audio latents from combined Pass-1 output
    sep_p1 = sep_av.EXECUTE_NORMALIZED(
        av_latent = get_value_at_index(sample_p1, 0)
    )
    video_lat_p1  = get_value_at_index(sep_p1, 0)
    audio_lat_p1  = get_value_at_index(sep_p1, 1)

    # ═══════════════════════════════════════════════════════════════════════
    #  BETWEEN PASSES  —  Crop guides → Upscale → Re-condition image
    # ═══════════════════════════════════════════════════════════════════════
    print('  🔧 Applying crop guides...')
    cropped = crop_guides.EXECUTE_NORMALIZED(
        positive = pos_cond,
        negative = neg_cond,
        latent   = video_lat_p1,
    )
    pos_cropped = get_value_at_index(cropped, 0)
    neg_cropped = get_value_at_index(cropped, 1)
    lat_cropped = get_value_at_index(cropped, 2)

    # Load VAE for upsampler + Pass-2 conditioning
    vae_p2  = vae_loader.load_vae(vae_name=vae_model_name)

    print('  🔍 Loading latent upscale model...')
    upscale_model = upscale_loader.EXECUTE_NORMALIZED(model_name=upscaler_name)

    print('  ⬆️  Upsampling latent x2...')
    upsampled = upsampler.upsample_latent(
        samples       = lat_cropped,
        upscale_model = get_value_at_index(upscale_model, 0),
        vae           = get_value_at_index(vae_p2, 0),
    )

    del upscale_model
    clear_gpu_memory()

    # Re-apply image conditioning on the upsampled latent
    img2vid_p2 = img2vid.EXECUTE_NORMALIZED(
        strength = image_strength,
        bypass   = image_bypass,
        vae      = get_value_at_index(vae_p2, 0),
        image    = get_value_at_index(preprocessed_image, 0),
        latent   = get_value_at_index(upsampled, 0),
    )

    del vae_p2
    clear_gpu_memory()

    # Concatenate upsampled video with Pass-1 audio latent
    video_for_p2 = (
        get_value_at_index(img2vid_p2, 0)
        if not image_bypass
        else get_value_at_index(upsampled, 0)
    )
    av_latent_p2 = concat_av.EXECUTE_NORMALIZED(
        video_latent = video_for_p2,
        audio_latent = audio_lat_p1,
    )

    print('  ✅ Upsampled latent ready for Pass-2.')
    log_memory('pre_pass2')

    # ═══════════════════════════════════════════════════════════════════════
    #  PASS 2  —  Refinement
    # ═══════════════════════════════════════════════════════════════════════
    print('  🎲 Pass-2 sampling (gradient_estimation)...')

    guider_p2 = cfg_guider.EXECUTE_NORMALIZED(
        cfg      = cfg,
        model    = model,
        positive = pos_cropped,
        negative = neg_cropped,
    )

    monitor_vram_guard(min_free_gb=1.0)

    sample_p2 = sampler_node.EXECUTE_NORMALIZED(
        noise        = get_value_at_index(noise_p2, 0),
        guider       = get_value_at_index(guider_p2, 0),
        sampler      = get_value_at_index(sampler_p2_obj, 0),
        sigmas       = get_value_at_index(sigmas_p2_obj, 0),
        latent_image = get_value_at_index(av_latent_p2, 0),
    )

    del guider_p2, model
    clear_gpu_memory()
    print('  ✅ Pass-2 complete.')
    log_memory('pass2_done')

    # Separate final AV latents (use output index 1 = denoised_output for P2)
    sep_p2 = sep_av.EXECUTE_NORMALIZED(
        av_latent = get_value_at_index(sample_p2, 1)
    )
    video_lat_final = get_value_at_index(sep_p2, 0)
    audio_lat_final = get_value_at_index(sep_p2, 1)

    return video_lat_final, audio_lat_final


print('🎲 Dual-pass sampler defined.')
print('\n✅ Section 13 complete.')

# 🎞️ Section 14 · VAE Decoder

In [ ]:
# @title 🎞️ 14. VAE Decoder { display-mode: "form" }
# @markdown VAEDecode (video) + LTXVAudioVAEDecode (audio) + CreateVideo.
# @markdown Mirrors workflow nodes 1, 24, and CreateVideo exactly.


def decode_latents(
    video_lat_final,
    audio_lat_final,
    vae_model_name: str,
    audio_vae,
    fps: int,
) -> tuple:
    """
    Decode video and audio latents to pixel-space frames.

    Pipeline:
      VAELoader(video_vae) → VAEDecode(video_latent) → frames tensor
      LTXVAudioVAEDecode(audio_latent, audio_vae) → audio tensor
      CreateVideo(fps, frames, audio) → video object

    The video VAE is loaded fresh here (freed after Pass-2) and deleted
    immediately after decoding to reclaim VRAM.

    Args:
        video_lat_final : Video latent from Pass-2.
        audio_lat_final : Audio latent from Pass-2.
        vae_model_name  : Video VAE filename.
        audio_vae       : Already-loaded audio VAE tuple.
        fps             : Output frame rate.

    Returns:
        (video_object, frames_tensor, audio_tensor)
    """
    ncm = NODE_CLASS_MAPPINGS

    # ── Video decode ───────────────────────────────────────────────────────
    print('  🎞️  Decoding video latent...')
    vae_loader = ncm['VAELoader']()
    vae        = vae_loader.load_vae(vae_name=vae_model_name)

    vae_decode  = ncm['VAEDecode']()
    decoded_vid = vae_decode.decode(
        samples = video_lat_final,
        vae     = get_value_at_index(vae, 0),
    )
    frames = get_value_at_index(decoded_vid, 0)  # (N, H, W, C) float32

    del vae
    clear_gpu_memory()
    print(f'  ✅ Video decoded: {frames.shape} frames.')

    # ── Audio decode ───────────────────────────────────────────────────────
    print('  🔊 Decoding audio latent...')
    audio_decode = ncm['LTXVAudioVAEDecode']()
    decoded_aud  = audio_decode.EXECUTE_NORMALIZED(
        samples   = audio_lat_final,
        audio_vae = get_value_at_index(audio_vae, 0),
    )
    audio = get_value_at_index(decoded_aud, 0)

    del audio_vae
    clear_gpu_memory()
    print('  ✅ Audio decoded.')

    # ── CreateVideo ────────────────────────────────────────────────────────
    print('  🎬 Creating video object...')
    create_video = ncm['CreateVideo']()
    video_obj    = create_video.EXECUTE_NORMALIZED(
        fps    = fps,
        images = frames,
        audio  = audio,
    )
    video = get_value_at_index(video_obj, 0)
    print('  ✅ Video object created.')
    log_memory('post_decode')

    return video, frames, audio


print('🎞️ Decoder defined.')
print('\n✅ Section 14 complete.')

# 💾 Section 15 · Video Export & Metadata

In [ ]:
# @title 💾 15. Video Export & Metadata { display-mode: "form" }
# @markdown Saves MP4, optional GIF preview, first-frame PNG, metadata JSON,
# @markdown and generation log to OUTPUT_FOLDER.

import datetime


def save_video_mp4(
    video_object,
    output_folder: str,
    prefix:        str = 'LTX2_SVI',
    fps:           int = 25,
) -> str:
    """
    Save a ComfyUI video object to an MP4 file.

    Uses folder_paths.get_save_image_path for auto-incrementing filename.
    Falls back to manual timestamp path if folder_paths is unavailable.

    Returns:
        Absolute path to the saved MP4.
    """
    from comfy_api.latest import Types
    import folder_paths

    w, h = video_object.get_dimensions()
    Path(output_folder).mkdir(parents=True, exist_ok=True)

    try:
        full_dir, fname, counter, _, _ = folder_paths.get_save_image_path(
            prefix, output_folder, w, h
        )
        ext  = Types.VideoContainer.get_extension('auto')
        path = os.path.join(full_dir, f'{fname}_{counter:05d}_.{ext}')
    except Exception:
        ts   = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        path = os.path.join(output_folder, f'{prefix}_{ts}.mp4')

    video_object.save_to(
        path,
        format   = Types.VideoContainer('auto'),
        codec    = 'auto',
        metadata = None,
    )
    print(f'  💾 MP4 saved: {path}')
    return path


def save_first_frame(
    frames,            # (N, H, W, C) tensor, float32 [0, 1]
    output_folder: str,
    prefix:        str = 'LTX2_SVI',
) -> str:
    """Save the first decoded frame as a PNG."""
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    frame0 = frames[0].cpu().numpy()          # (H, W, C)
    img    = Image.fromarray((frame0 * 255).astype('uint8'))
    ts     = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    path   = os.path.join(output_folder, f'{prefix}_{ts}_frame0.png')
    img.save(path)
    print(f'  🖼️  First frame: {path}')
    return path


def save_preview_gif(
    frames,
    output_folder: str,
    prefix:        str  = 'LTX2_SVI',
    fps:           int  = 10,
    max_side:      int  = 320,
) -> str:
    """
    Save a low-res preview GIF (every 2nd frame, max 320px wide).

    GIF is for quick sanity-check only — not for distribution.
    """
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    ts    = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    path  = os.path.join(output_folder, f'{prefix}_{ts}_preview.gif')

    imgs  = []
    # Every 2nd frame
    for i in range(0, len(frames), 2):
        f   = frames[i].cpu().numpy()
        img = Image.fromarray((f * 255).astype('uint8'))
        # Downscale to max_side
        ratio = max_side / max(img.width, img.height)
        if ratio < 1:
            img = img.resize(
                (int(img.width * ratio), int(img.height * ratio)),
                Image.LANCZOS,
            )
        imgs.append(img)

    if imgs:
        imgs[0].save(
            path, save_all=True, append_images=imgs[1:],
            duration=int(1000 / fps), loop=0,
        )
        print(f'  🎞️  Preview GIF: {path}')

    return path


def save_metadata_json(
    output_folder:  str,
    prefix:         str,
    scene:          'Scene',
    compiled_pos:   str,
    compiled_neg:   str,
    seed:           int,
    width:          int,
    height:         int,
    fps:            int,
    frame_count:    int,
    elapsed_s:      float,
    mp4_path:       str,
    frame0_path:    str = '',
    gif_path:       str = '',
    extra:          dict = None,
) -> str:
    """Write generation metadata as a JSON sidecar file."""
    Path(output_folder).mkdir(parents=True, exist_ok=True)
    ts   = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    path = os.path.join(output_folder, f'{prefix}_{ts}_meta.json')

    meta = {
        'generated_at'   : ts,
        'model'          : DIT_MODEL,
        'text_encoder_1' : TEXT_ENC1_MODEL,
        'text_encoder_2' : TEXT_ENC2_MODEL,
        'vae_video'      : VAE_VIDEO_MODEL,
        'vae_audio'      : VAE_AUDIO_MODEL,
        'upscaler'       : UPSCALER_MODEL,
        'scene_id'       : scene.scene_id,
        'prompt_raw'     : scene.prompt,
        'prompt_compiled': compiled_pos,
        'negative_prompt': compiled_neg,
        'seed'           : seed,
        'width'          : width,
        'height'         : height,
        'fps'            : fps,
        'frame_count'    : frame_count,
        'duration_s'     : frame_count / fps,
        'cfg'            : CFG_SCALE,
        'sampler_p1'     : SAMPLER_PASS1,
        'sampler_p2'     : SAMPLER_PASS2,
        'sigmas_p1'      : SIGMAS_PASS1,
        'sigmas_p2'      : SIGMAS_PASS2,
        'lora_1'         : LORA_1,
        'lora_2'         : LORA_2,
        'lora_3'         : LORA_3,
        'elapsed_seconds': round(elapsed_s, 1),
        'output_mp4'     : mp4_path,
        'output_frame0'  : frame0_path,
        'output_gif'     : gif_path,
    }
    if extra:
        meta.update(extra)

    with open(path, 'w') as f:
        json.dump(meta, f, indent=2)
    print(f'  📋 Metadata: {path}')
    return path


print('💾 Export functions defined.')
print('\n✅ Section 15 complete.')

# 👁️ Section 16 · Preview

In [ ]:
# @title 👁️ 16. Preview Utilities { display-mode: "form" }
# @markdown Inline video display, first-frame viewer, and upload helpers.

from IPython.display import display, HTML, Image as IPImage
from base64 import b64encode


def display_video(video_path: str, width: int = 512) -> None:
    """
    Render an MP4 or WebM inline in the Colab notebook.

    Uses a base64 data URI — works without a running HTTP server.
    """
    if not Path(video_path).exists():
        print(f'⚠️  Video not found: {video_path}')
        return

    ext  = Path(video_path).suffix.lower()
    mime = {
        '.mp4' : 'video/mp4',
        '.webm': 'video/webm',
        '.mov' : 'video/quicktime',
    }.get(ext, 'video/mp4')

    data = open(video_path, 'rb').read()
    b64  = b64encode(data).decode()
    display(HTML(
        f'<video width="{width}" controls autoplay loop muted>'
        f'  <source src="data:{mime};base64,{b64}" type="{mime}">'
        f'</video>'
    ))


def display_image(image_path: str) -> None:
    """Display a PNG/JPG inline."""
    if not Path(image_path).exists():
        print(f'⚠️  Image not found: {image_path}')
        return
    display(IPImage(filename=image_path))


def display_generation_summary(
    scene:        'Scene',
    mp4_path:     str,
    frame0_path:  str,
    elapsed_s:    float,
    compiled_pos: str,
) -> None:
    """Print a formatted generation summary."""
    print()
    print('╔' + '═' * 58 + '╗')
    print('║  🎬 GENERATION COMPLETE' + ' ' * 34 + '║')
    print('╠' + '═' * 58 + '╣')
    print(f'║  Scene      : {str(scene.scene_id):41s} ║')
    print(f'║  Duration   : {elapsed_s:.0f}s elapsed' + ' ' * (40 - len(f'{elapsed_s:.0f}s elapsed')) + ' ║')
    print(f'║  Output     : {mp4_path[-40:]:41s} ║')
    print('╚' + '═' * 58 + '╝')
    print()


def upload_image_to_input() -> Optional[str]:
    """
    Upload an image from the local machine to /content/ComfyUI/input/.

    Returns the full path of the uploaded file, or None.
    """
    from google.colab import files
    os.makedirs(str(COMFYUI_DIR / 'input'), exist_ok=True)
    uploaded = files.upload()
    for fname in uploaded.keys():
        src  = f'/content/ComfyUI/{fname}'
        dest = str(COMFYUI_DIR / 'input' / fname)
        try:
            shutil.move(src, dest)
        except Exception:
            dest = src
        print(f'  📁 Saved to: {dest}')
        return dest
    return None


def download_outputs(output_folder: str = OUTPUT_FOLDER) -> None:
    """Download all files from the output folder to the local machine."""
    from google.colab import files
    for f in sorted(Path(output_folder).glob('*')):
        if f.is_file():
            print(f'  ⬇️  Downloading: {f.name}')
            files.download(str(f))


print('👁️  Preview utilities defined.')
print('\n✅ Section 16 complete.')

# 🧹 Section 17 · Cleanup

In [ ]:
# @title 🧹 17. Cleanup { display-mode: "form" }
# @markdown Utility functions to free VRAM/RAM after generation.
# @markdown Run this cell between generations or when VRAM is tight.


def full_cleanup(verbose: bool = True) -> None:
    """
    Aggressive memory cleanup:
      - torch.cuda.empty_cache()
      - gc.collect() × 3
      - torch.cuda.ipc_collect()
      - Delete any lingering tensor globals
    """
    gc.collect()
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    gc.collect()
    if verbose:
        print('🧹 Memory cleaned.')
        log_memory('after_cleanup')


def unload_comfyui_models() -> None:
    """
    Signal ComfyUI's model manager to offload cached models.

    Useful when switching between scenes to free full VRAM.
    """
    try:
        import comfy.model_management as model_management
        model_management.unload_all_models()
        model_management.soft_empty_cache()
        print('  ✅ ComfyUI models unloaded.')
    except Exception as e:
        print(f'  ⚠️  ComfyUI unload skipped: {e}')
    full_cleanup(verbose=False)


def cleanup_output_folder(keep_latest: int = 5, output_folder: str = OUTPUT_FOLDER) -> None:
    """
    Remove older output files, keeping only the `keep_latest` most recent MP4s.

    JSON and PNG sidecars are kept to preserve metadata.
    """
    mp4s = sorted(Path(output_folder).glob('*.mp4'), key=os.path.getmtime)
    to_delete = mp4s[:-keep_latest] if len(mp4s) > keep_latest else []
    for p in to_delete:
        p.unlink()
        print(f'  🗑️  Removed: {p.name}')
    if not to_delete:
        print(f'  ✅ No old MP4s to remove (≤{keep_latest} present).')


print('🧹 Cleanup utilities defined.')
print('\n✅ Section 17 complete.')
print('\n' + '═' * 60)
print('  All pipeline sections loaded successfully.')
print('  Proceed to Section 18 → Upload Image & Run Generation.')
print('═' * 60)

# 🚀 Section 18 · Upload Image & Run Generation

**This is the main execution cell.**

1. (Optional) Upload a reference image below — or leave blank for Text-to-Video.
2. Configure the prompt and settings.
3. Run the cell.

All previous sections must have been run first.

In [ ]:
# @title 📷 Upload Reference Image (Optional) { display-mode: "form" }
# @markdown Upload an image for Image-to-Video. Skip to use Text-to-Video mode.
# @markdown You can also set `UPLOADED_IMAGE_PATH` manually to a path string.

UPLOADED_IMAGE_PATH: Optional[str] = None

do_upload = False  # @param {type:"boolean"}
display_after_upload = True  # @param {type:"boolean"}

if do_upload:
    UPLOADED_IMAGE_PATH = upload_image_to_input()
    if UPLOADED_IMAGE_PATH and display_after_upload:
        display_image(UPLOADED_IMAGE_PATH)

print(f'Image path: {UPLOADED_IMAGE_PATH or "(none — Text-to-Video mode)"}')

In [ ]:
# @title 🚀 Generate Video { display-mode: "form" }
# @markdown ┌─────────────────────────────────────────────────────────────┐
# @markdown │  QUICK PROMPT OVERRIDE  (leave blank to use story scene)  │
# @markdown └─────────────────────────────────────────────────────────────┘

QUICK_PROMPT    = ''  # @param {type:"string"}
QUICK_NEG       = ''  # @param {type:"string"}
QUICK_SEED      = -1  # @param {type:"integer"} — use -1 to inherit SEED
QUICK_WIDTH     = -1  # @param {type:"integer"} — use -1 to inherit OUTPUT_WIDTH
QUICK_HEIGHT    = -1  # @param {type:"integer"} — use -1 to inherit OUTPUT_HEIGHT
QUICK_DURATION  = -1  # @param {type:"number"}  — use -1 to inherit DURATION_SECONDS
QUICK_FPS       = -1  # @param {type:"integer"} — use -1 to inherit FPS

# @markdown ---
# @markdown **LoRA toggles** (models must have been downloaded in Section 5)
USE_LORA_1  = False  # @param {type:"boolean"}
USE_LORA_2  = False  # @param {type:"boolean"}
USE_LORA_3  = False  # @param {type:"boolean"}

# @markdown ---
# @markdown **Image override** (overrides uploaded image if set)
IMAGE_PATH_OVERRIDE = ''  # @param {type:"string"}
DISABLE_IMAGE       = False  # @param {type:"boolean"} — force T2V even if image present

# ══════════════════════════════════════════════════════════════════════════


def generate_video(
    scene:               'Scene',
    image_path:          Optional[str]  = None,
    override_prompt:     str            = '',
    override_neg:        str            = '',
    override_seed:       int            = -1,
    override_width:      int            = -1,
    override_height:     int            = -1,
    override_fps:        int            = -1,
    override_duration:   float          = -1,
    use_lora_1:          bool           = False,
    use_lora_2:          bool           = False,
    use_lora_3:          bool           = False,
    auto_display:        bool           = True,
) -> dict:
    """
    Main orchestration function: end-to-end video generation.

    Calls every pipeline stage in order:
      Sec 6  → ensure_nodes_loaded()
      Sec 10 → compile_prompt()
      Sec 12 → build_image_latent()
      Sec 13 → run_text_encoding() + run_dual_pass_sampling()
      Sec 14 → decode_latents()
      Sec 15 → save_video_mp4(), save_first_frame(), save_metadata_json()
      Sec 16 → display_video()
      Sec 17 → full_cleanup()

    Args:
        scene            : Scene dataclass (from STORY_SCENES).
        image_path       : Override image path; falls back to scene.image_path.
        override_*       : Quick overrides for individual parameters.
        use_lora_*       : Enable LoRA 1/2/3.
        auto_display     : Auto-display video after generation.

    Returns:
        Dict with keys: mp4_path, frame0_path, meta_path, elapsed_s.
    """
    t_start = time.time()

    # ── Resolve parameters (overrides > scene > global config) ────────────
    seed       = override_seed    if override_seed    >= 0 else (scene.seed    or SEED)
    width      = override_width   if override_width   > 0  else (scene.width   or OUTPUT_WIDTH)
    height     = override_height  if override_height  > 0  else (scene.height  or OUTPUT_HEIGHT)
    fps        = override_fps     if override_fps     > 0  else (scene.fps     or FPS)
    duration   = override_duration if override_duration > 0 else scene.duration_seconds

    # Hardware safety clamps
    width      = min(width,    MAX_WIDTH_T4)
    height     = min(height,   MAX_HEIGHT_T4)
    duration   = min(duration, MAX_DURATION)
    fps        = min(fps,      MAX_FPS)

    raw_frames   = int(duration * fps)
    frame_count  = max(((raw_frames - 1) // 8) * 8 + 1, 9)
    latent_w     = width  // 2
    latent_h     = height // 2

    # Resolve image path
    _img_path = image_path or scene.image_path
    if not _img_path or not Path(_img_path).exists():
        _img_path = None

    # Compile prompts
    if override_prompt:
        pos_text = override_prompt
        neg_text = override_neg or compile_negative_prompt(scene, CHARACTER_DB)
    else:
        pos_text = compile_prompt(scene, CHARACTER_DB, GLOBAL_DIRECTOR_PROMPT)
        neg_text = compile_negative_prompt(scene, CHARACTER_DB)

    print('\n' + '═' * 60)
    print(f'  🎬 GENERATING  Scene {scene.scene_id}')
    print('═' * 60)
    print(f'  Size     : {width}×{height}  |  {frame_count} frames  |  {fps} fps')
    print(f'  Seed     : {seed}')
    print(f'  Image    : {_img_path or "T2V (no image)"}')
    print(f'  Prompt   : {pos_text[:100]}...')
    log_memory('start')

    # ── Load ComfyUI nodes (idempotent) ────────────────────────────────────
    ensure_nodes_loaded()

    results = {}

    with torch.inference_mode():
        try:
            # ── STAGE 1: Build latent ──────────────────────────────────────
            print('\n[Stage 1/4] Building image latent...')
            (
                av_latent,
                preprocessed_img,
                image_bypass,
                image_strength_eff,
                audio_vae,
                empty_latent,
            ) = build_image_latent(
                image_path      = _img_path,
                width           = width,
                height          = height,
                frame_count     = frame_count,
                fps             = fps,
                img_compression = IMG_COMPRESSION,
                longer_edge     = LONGER_EDGE,
                image_strength  = IMAGE_STRENGTH,
                vae_model_name  = VAE_VIDEO_MODEL,
                audio_vae_name  = VAE_AUDIO_MODEL,
                latent_w        = latent_w,
                latent_h        = latent_h,
            )

            # ── STAGE 2: Text encoding ─────────────────────────────────────
            print('\n[Stage 2/4] Text encoding...')
            pos_cond, neg_cond = run_text_encoding(
                prompt     = pos_text,
                neg_prompt = neg_text,
                fps        = fps,
            )

            # ── Load DiT ───────────────────────────────────────────────────
            dit_model = load_dit_model(
                lora_1 = LORA_1 if use_lora_1 else None,
                lora_2 = LORA_2 if use_lora_2 else None,
                lora_3 = LORA_3 if use_lora_3 else None,
                lora_s1= LORA_1_STRENGTH,
                lora_s2= LORA_2_STRENGTH,
                lora_s3= LORA_3_STRENGTH,
            )

            # ── STAGE 3: Dual-pass sampling ────────────────────────────────
            print('\n[Stage 3/4] Dual-pass sampling...')
            video_lat, audio_lat = run_dual_pass_sampling(
                pos_cond        = pos_cond,
                neg_cond        = neg_cond,
                av_latent       = av_latent,
                preprocessed_image = preprocessed_img,
                audio_vae       = audio_vae,
                image_bypass    = image_bypass,
                image_strength  = image_strength_eff,
                seed            = seed,
                cfg             = CFG_SCALE,
                sampler_p1      = SAMPLER_PASS1,
                sampler_p2      = SAMPLER_PASS2,
                sigmas_p1       = SIGMAS_PASS1,
                sigmas_p2       = SIGMAS_PASS2,
                vae_model_name  = VAE_VIDEO_MODEL,
                upscaler_name   = UPSCALER_MODEL,
                fps             = fps,
                frame_count     = frame_count,
                latent_w        = latent_w,
                latent_h        = latent_h,
                model           = dit_model,
            )

            del dit_model, pos_cond, neg_cond, av_latent
            clear_gpu_memory()

            # ── STAGE 4: Decode & export ───────────────────────────────────
            print('\n[Stage 4/4] Decoding & exporting...')
            video_obj, frames, audio = decode_latents(
                video_lat_final = video_lat,
                audio_lat_final = audio_lat,
                vae_model_name  = VAE_VIDEO_MODEL,
                audio_vae       = audio_vae,
                fps             = fps,
            )

            elapsed = time.time() - t_start

            mp4_path = save_video_mp4(
                video_object   = video_obj,
                output_folder  = OUTPUT_FOLDER,
                prefix         = OUTPUT_PREFIX,
                fps            = fps,
            )

            frame0_path = ''
            if SAVE_FIRST_FRAME:
                frame0_path = save_first_frame(frames, OUTPUT_FOLDER, OUTPUT_PREFIX)

            gif_path = ''
            if SAVE_PREVIEW_GIF:
                gif_path = save_preview_gif(frames, OUTPUT_FOLDER, OUTPUT_PREFIX, fps=10)

            meta_path = save_metadata_json(
                output_folder  = OUTPUT_FOLDER,
                prefix         = OUTPUT_PREFIX,
                scene          = scene,
                compiled_pos   = pos_text,
                compiled_neg   = neg_text,
                seed           = seed,
                width          = width,
                height         = height,
                fps            = fps,
                frame_count    = frame_count,
                elapsed_s      = elapsed,
                mp4_path       = mp4_path,
                frame0_path    = frame0_path,
                gif_path       = gif_path,
            ) if SAVE_METADATA else ''

            results = {
                'mp4_path'  : mp4_path,
                'frame0'    : frame0_path,
                'meta'      : meta_path,
                'elapsed_s' : elapsed,
            }

            display_generation_summary(scene, mp4_path, frame0_path, elapsed, pos_text)

            if auto_display:
                display_video(mp4_path)

        except torch.cuda.OutOfMemoryError:
            print('\n❌ CUDA OUT OF MEMORY')
            print('   Suggestions:')
            print('   • Reduce OUTPUT_WIDTH / OUTPUT_HEIGHT')
            print('   • Reduce FRAME_COUNT or DURATION_SECONDS')
            print('   • Run Section 17 cleanup cell and retry')
            full_cleanup()
            raise

        except Exception as exc:
            print(f'\n❌ Generation failed: {exc}')
            import traceback
            traceback.print_exc()
            full_cleanup()
            raise

    full_cleanup(verbose=False)
    return results


# ══════════════════════════════════════════════════════════════════════════
#  EXECUTE GENERATION
# ══════════════════════════════════════════════════════════════════════════

# Select the active scene
_active_scene = STORY_SCENES[ACTIVE_SCENE_INDEX]

# Apply image override
_image_path = (
    IMAGE_PATH_OVERRIDE if IMAGE_PATH_OVERRIDE
    else (UPLOADED_IMAGE_PATH if not DISABLE_IMAGE else None)
)

generation_results = generate_video(
    scene             = _active_scene,
    image_path        = _image_path,
    override_prompt   = QUICK_PROMPT,
    override_neg      = QUICK_NEG,
    override_seed     = QUICK_SEED,
    override_width    = QUICK_WIDTH,
    override_height   = QUICK_HEIGHT,
    override_fps      = QUICK_FPS,
    override_duration = QUICK_DURATION,
    use_lora_1        = USE_LORA_1,
    use_lora_2        = USE_LORA_2,
    use_lora_3        = USE_LORA_3,
    auto_display      = True,
)

# 🔄 Section 19 · Multi-Scene Story Runner

Run all scenes in `STORY_SCENES` sequentially with automatic cleanup between clips.

In [ ]:
# @title 🔄 19. Multi-Scene Story Runner { display-mode: "form" }
# @markdown Run every scene in STORY_SCENES. Results are saved to OUTPUT_FOLDER.

RUN_ALL_SCENES   = False  # @param {type:"boolean"}
CLEANUP_BETWEEN  = True   # @param {type:"boolean"}
RETRY_ON_OOM     = True   # @param {type:"boolean"}

if RUN_ALL_SCENES:
    all_results = []
    print(f'🎬 Starting multi-scene story run ({len(STORY_SCENES)} scenes)...')

    for idx, scene in enumerate(STORY_SCENES):
        print(f'\n[{idx+1}/{len(STORY_SCENES)}] Scene {scene.scene_id}: {scene.prompt[:60]}...')

        # Use character reference image if available
        ref_image = CHARACTER_DB.get_reference_image(scene.character_ids)

        success = False
        for attempt in range(1, (3 if RETRY_ON_OOM else 2)):
            try:
                res = generate_video(
                    scene      = scene,
                    image_path = ref_image,
                    use_lora_1 = USE_LORA_1,
                    use_lora_2 = USE_LORA_2,
                    use_lora_3 = USE_LORA_3,
                    auto_display = True,
                )
                all_results.append(res)
                success = True
                break
            except torch.cuda.OutOfMemoryError:
                print(f'  ⚠️  OOM on attempt {attempt}. Cleaning memory...')
                full_cleanup()
                unload_comfyui_models()
                time.sleep(5)

        if not success:
            print(f'  ❌ Scene {scene.scene_id} failed after retries. Skipping.')

        if CLEANUP_BETWEEN:
            full_cleanup()
            if hasattr(torch.cuda, 'empty_cache'):
                torch.cuda.empty_cache()

    print(f'\n✅ Story run complete: {len(all_results)}/{len(STORY_SCENES)} scenes generated.')
    print(f'   Output folder: {OUTPUT_FOLDER}')

else:
    print('ℹ️  Set RUN_ALL_SCENES = True to run all story scenes.')

# 💾 Section 20 · Download Outputs & Cleanup Run

In [ ]:
# @title 💾 20. Download Outputs & Final Cleanup { display-mode: "form" }

DO_DOWNLOAD  = False  # @param {type:"boolean"}
DO_CLEANUP   = True   # @param {type:"boolean"}
KEEP_LATEST  = 5      # @param {type:"integer"}

if DO_CLEANUP:
    full_cleanup()
    unload_comfyui_models()
    cleanup_output_folder(keep_latest=KEEP_LATEST)

if DO_DOWNLOAD:
    print('⬇️  Downloading all output files...')
    download_outputs(OUTPUT_FOLDER)

# List all outputs
print(f'\n📁 Files in {OUTPUT_FOLDER}:')
for f in sorted(Path(OUTPUT_FOLDER).glob('*')):
    size_mb = f.stat().st_size / 1e6
    print(f'   {f.name:50s}  {size_mb:.1f} MB')